# Spec 撰寫模板 (Spec Writing Template)

## 模組脈絡：意圖收斂的終點——把需求寫成「規格契約」

本筆記是 **02-意圖收斂** 的 capstone。前面學了各種 prompt 技巧；這一章把它們**系統化**成一份可重用的**規格（spec）**。spec 是你與模型之間的契約：把模糊的「幫我寫個東西」收斂成角色、目標、輸入、限制、輸出格式、成功標準都明確的指令。

**為什麼重要**：spec 把「不可控」收斂到最低——同一份 spec 給不同模型（OpenAI/Claude/Gemini）都應產生結構一致、可驗收的輸出。這也是後續模組（結構化輸出、agent、評估）的共同起點。

## 1. Spec 的七個欄位

一份好的 spec 至少涵蓋：

| 欄位 | 作用 | 對應可控性 |
|------|------|-----------|
| **Role** 角色 | 設定專業視角與語氣 | 收斂風格 |
| **Goal** 目標 | 一句話說清要產出什麼 | 收斂意圖 |
| **Inputs** 輸入 | 明確列出可用素材 | 減少臆測 |
| **Constraints** 限制 | 字數、語言、禁止事項 | 收斂行為邊界 |
| **Output format** 輸出格式 | JSON/Markdown/結構 | 收斂結構（接模組 03） |
| **Success criteria** 成功標準 | 可驗收的條件 | 收斂評估（接模組 07） |
| **Examples** 範例 | few-shot 示範 | 收斂風格與格式 |

## 2. 環境設定

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()  # 讀取 OPENAI_API_KEY

## 3. 可重用的 Spec 模板

用 Python 函式把七欄位組成一個結構化 system prompt——這就是你的「spec 產生器」。

In [ ]:
def build_spec(role, goal, inputs, constraints, output_format, success_criteria, examples=None):
    """把需求七要素組成一份結構化 spec（system prompt）。"""
    parts = [
        f"# 角色\n{role}",
        f"# 目標\n{goal}",
        f"# 可用輸入\n{inputs}",
        f"# 限制\n{constraints}",
        f"# 輸出格式\n{output_format}",
        f"# 成功標準\n{success_criteria}",
    ]
    if examples:
        parts.append(f"# 範例\n{examples}")
    return "\n\n".join(parts)

spec = build_spec(
    role="你是一位資深 DTC 電商文案，擅長精簡有力的產品標題。",
    goal="為輸入的產品，產生 3 個吸睛的繁體中文標題。",
    inputs="使用者會提供產品名稱與賣點。",
    constraints="每個標題 ≤ 20 字；不得誇大療效；繁體中文。",
    output_format='回傳 JSON：{"titles": ["...", "...", "..."]}',
    success_criteria="3 個標題、皆 ≤20 字、皆與賣點相關。",
    examples='產品：保溫瓶 / 賣點：12 小時保溫 → {"titles": ["12 小時還燙口", ...]}',
)
print(spec)

## 4. 用 spec 驅動模型（OpenAI）

spec 當 system message，使用者只需丟最精簡的輸入。輸出受 spec 約束、可直接解析。

In [ ]:
import os, json
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

def run_with_spec_openai(spec, user_input, model=OPENAI_MODEL):
    resp = openai_client.responses.create(
        model=model,
        input=[{"role": "system", "content": spec},
               {"role": "user", "content": user_input}],
        temperature=0.7,
        text={"format": {"type": "json_object"}},
    )
    return json.loads(resp.output_text)

out = run_with_spec_openai(spec, "產品：人體工學椅 / 賣點：久坐不腰痛、可調腰靠")
print(out)

## 5. 同一份 spec 跨模型一致性（Claude / Gemini）

好的 spec 應該是**模型無關**的——換 provider 仍得到結構一致的輸出。這是「收斂」的最佳證明。

In [ ]:
# Claude
try:
    import anthropic
    claude = anthropic.Anthropic()
    msg = claude.messages.create(
        model="claude-sonnet-4-6", max_tokens=512, system=spec,
        messages=[{"role": "user", "content": "產品：人體工學椅 / 賣點：久坐不腰痛、可調腰靠"}],
    )
    print("Claude:", msg.content[0].text)
except Exception as e:
    print("跳過 Claude：", type(e).__name__, e)

In [ ]:
# Gemini（用 response_schema 強制 JSON 結構）
try:
    from google import genai
    from google.genai import types
    gem = genai.Client()
    resp = gem.models.generate_content(
        model="gemini-2.5-flash",
        contents="產品：人體工學椅 / 賣點：久坐不腰痛、可調腰靠",
        config=types.GenerateContentConfig(
            system_instruction=spec,
            response_mime_type="application/json",
        ),
    )
    print("Gemini:", resp.text)
except Exception as e:
    print("跳過 Gemini：", type(e).__name__, e)

## 6. 練習

用 `build_spec()` 為下列任務各寫一份 spec，並比較有無 spec 的輸出差異：

1. 把一段會議記錄整理成「決議 / 待辦 / 負責人」三欄表格。
2. 將技術部落格文章改寫成給高中生看的科普短文（≤300 字）。

> 觀察重點：spec 是否讓**不同次執行、不同模型**的輸出結構穩定一致？

---

## 本章小結

1. **spec = 模型契約**：role / goal / inputs / constraints / output format / success criteria / examples。
2. spec 把模組 02 的所有手法（角色、步驟、few-shot、格式）**系統化**成可重用模板。
3. 好 spec 是**模型無關**的，OpenAI / Claude / Gemini 都應產生結構一致的輸出。
4. spec 的「輸出格式」與「成功標準」直接銜接模組 03（結構化輸出）與模組 07（評估）。
5. **收斂的終點**：把模糊需求壓成可驗收的規格，是把不可控變可控的核心工程動作。